<a href="https://colab.research.google.com/github/JozefSL/pyNotes/blob/main/Kalman/State2Region.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Kalman Filter Implementation Template

This template provides a basic structure for a Kalman filter. You'll need to adapt the state space model (A, B, H matrices), covariance matrices (Q, R), and initial conditions to your specific oil and gas production forecasting problem, considering your predicted well count, estimated production for new wells, and legacy decline factors.

In [1]:
import numpy as np

def kalman_filter(observations, initial_state_estimate, initial_covariance_estimate, A, B, H, Q, R, control_input=None):
    """
    Implements a basic Kalman filter.

    Args:
        observations (list or np.array): A sequence of measurements.
        initial_state_estimate (np.array): Initial belief about the system's state.
        initial_covariance_estimate (np.array): Initial uncertainty in the state estimate.
        A (np.array): State transition matrix.
        B (np.array): Control-input matrix (optional, set to identity or zero if no control input).
        H (np.array): Observation matrix.
        Q (np.array): Process noise covariance matrix.
        R (np.array): Measurement noise covariance matrix.
        control_input (list or np.array, optional): A sequence of control inputs for each time step.

    Returns:
        tuple: A tuple containing lists of state estimates and their covariances.
    """
    num_states = initial_state_estimate.shape[0]
    num_observations = observations[0].shape[0]

    # Initialize state and covariance
    current_state_estimate = initial_state_estimate
    current_covariance_estimate = initial_covariance_estimate

    # Lists to store results
    state_estimates = []
    covariance_estimates = []

    for i, z_k in enumerate(observations):
        # --- Prediction Step ---
        # Predict next state
        if control_input is not None and B is not None:
            # Ensure control_input is a 2D array for matrix multiplication if B is 2D
            u_k = np.array(control_input[i]).reshape(-1, 1) if control_input[i] is not None else np.zeros((B.shape[1], 1))
            predicted_state = np.dot(A, current_state_estimate) + np.dot(B, u_k)
        else:
            predicted_state = np.dot(A, current_state_estimate)

        # Predict next covariance
        predicted_covariance = np.dot(np.dot(A, current_covariance_estimate), A.T) + Q

        # --- Update Step ---
        # Kalman Gain
        S_k = np.dot(np.dot(H, predicted_covariance), H.T) + R
        K_k = np.dot(np.dot(predicted_covariance, H.T), np.linalg.inv(S_k))

        # Update state estimate
        y_k = z_k - np.dot(H, predicted_state) # Measurement residual
        current_state_estimate = predicted_state + np.dot(K_k, y_k)

        # Update covariance estimate
        current_covariance_estimate = predicted_covariance - np.dot(np.dot(K_k, H), predicted_covariance)

        state_estimates.append(current_state_estimate)
        covariance_estimates.append(current_covariance_estimate)

    return state_estimates, covariance_estimates


#### **1. Define Your State Vector and Matrices**

Your `state_vector` will represent the variables you want to estimate. For oil and gas, this might include:
*   Total production for Permian, Eagle Ford, Haynesville.
*   Underlying true decline rates.
*   Potentially a bias term to account for systematic under/over-reporting.

`A` (State Transition Matrix): Describes how the state evolves from `t` to `t+1` in the absence of noise. This is where your existing forecast logic (predicted well count, new well production, legacy decline) comes in.

`B` (Control-Input Matrix): If you have external factors that deterministically influence the state (e.g., policy changes, known drilling schedules), this matrix applies a `control_input` to the state.

`H` (Observation Matrix): Relates the true state to your noisy measurements. This is crucial for handling the different reporting types (EIA 914, delayed well data).

In [2]:
# Example: Simple 1D state (e.g., total production for one basin)

# State: [production_t]
initial_state_estimate = np.array([[100.0]]) # Initial guess for production

# State Transition Matrix (A):
# Assuming production simply carries over, or with a simple decline factor.
# For your case, this would incorporate your forecast model.
# Example: Production_t+1 = 0.95 * Production_t (simple decline)
A = np.array([[0.95]])

# Control-Input Matrix (B): If you have external inputs
# If no control input, B can be None or an appropriately sized zero matrix.
# B = np.array([[1.0]]) # Example: control input directly adds to production
B = None

# Observation Matrix (H):
# If you measure the state directly (e.g., EIA data reports total production).
# If EIA 914 reports total production, H would be [[1.0]].
# If you have multiple measurements (e.g., EIA and then well-level later), H will change over time.
H = np.array([[1.0]])


#### **2. Define Covariance Matrices**

`Q` (Process Noise Covariance): Represents the uncertainty in your state transition model (your forecast). How much does your forecast deviate from the true underlying dynamics? If your forecast is very reliable, `Q` will be small.

`R` (Measurement Noise Covariance): Represents the uncertainty in your measurements. EIA 914 data might have a relatively small `R` (high confidence), while early well-level data (50% reported) would have a larger `R` (lower confidence).

In [3]:
# Initial Covariance Estimate (P):
# How uncertain are you about your initial_state_estimate?
initial_covariance_estimate = np.array([[10.0]])

# Process Noise Covariance (Q):
# Uncertainty in the model's prediction of the next state.
Q = np.array([[0.1]]) # Small value if your model is accurate

# Measurement Noise Covariance (R):
# Uncertainty in the observation (EIA 914 data, well-level data).
# This will likely change depending on the data source and its completeness.
R = np.array([[1.0]]) # Example: assuming some measurement noise


#### **3. Prepare Observations and Control Inputs**

Your `observations` will be your actual reported data. This is where you'll handle the different data sources and their varying latencies:

*   **Month 1:** You might use EIA 914 data as `z_k`.
*   **Month 2:** You might combine EIA 914 and partially reported well-level data. You'll need to decide how to construct `z_k` and `H` for this combined observation.
*   **Month 3+:** As more well-level data comes in, your `H` and `R` matrices will adapt to reflect the increasing accuracy and completeness of your observations.

`control_input` (optional): Any known deterministic changes that impact production, separate from the inherent state dynamics.

In [4]:
# Simulate some observations
# In your case, this would be your actual EIA 914 and well-level data
observations = [
    np.array([[99.0]]),
    np.array([[102.0]]),
    np.array([[101.5]]),
    np.array([[98.0]]),
    np.array([[97.5]])
]

# If you have control inputs, provide them here for each observation
control_inputs = [None] * len(observations)
# Example with control input:
# control_inputs = [np.array([[1.0]]), np.array([[0.5]]), None, np.array([[-0.2]]), None]


#### **4. Run the Kalman Filter**

Now you can call the `kalman_filter` function with your defined parameters.

In [5]:
state_estimates, covariance_estimates = kalman_filter(
    observations,
    initial_state_estimate,
    initial_covariance_estimate,
    A, B, H, Q, R,
    control_input=control_inputs
)

print("State Estimates:")
for i, estimate in enumerate(state_estimates):
    print(f"Time {i+1}: {estimate.flatten()[0]:.2f}")

print("\nCovariance Estimates (uncertainty):")
for i, cov in enumerate(covariance_estimates):
    print(f"Time {i+1}: {cov.flatten()[0]:.2f}")


State Estimates:
Time 1: 98.60
Time 2: 97.65
Time 3: 95.79
Time 4: 93.05
Time 5: 90.83

Covariance Estimates (uncertainty):
Time 1: 0.90
Time 2: 0.48
Time 3: 0.35
Time 4: 0.29
Time 5: 0.27


#### **Key Considerations for Your Problem:**

1.  **Dynamic `H` and `R`:** This is the most critical part for your situation. For each time step, `H` and `R` might change to reflect the available data:
    *   When only EIA 914 is available, `H` maps to this aggregated observation, and `R` reflects its uncertainty.
    *   As well-level data becomes available (e.g., 50% in month 1, 90% in month 3), you'll need to dynamically adjust `H` to map to these partial observations and `R` to reflect the increasing completeness and accuracy.
    *   You might even have multiple `H` and `R` matrices for different observation types if you process them simultaneously.
2.  **State Vector Complexity:** You might need to expand your state vector to include separate production values for each basin (Permian, Eagle Ford, Haynesville) and potentially even components that track reporting lag or bias.
3.  **Process Noise `Q`:** Tune `Q` to reflect the inherent uncertainty in your predictive well count, new well production, and legacy decline factors. A larger `Q` means the filter trusts your model less and relies more on measurements.
4.  **Measurement Noise `R`:** Carefully assign `R` values. Lower `R` (more trust) for EIA 914, and progressively lower `R` for well-level data as it becomes more complete.
5.  **Handling Zero Production/Missing Data:** Ensure your observation function handles periods where no data is available or when production is genuinely zero. Kalman filters are good at estimating through gaps.

This template provides the mathematical core. The real work will be in modeling your `A`, `H`, `Q`, and `R` matrices to accurately represent your specific data sources and forecasting logic over time.